# V4: eight-fit learning-rate diagnostic
For JAY and ARM at seeds 45 and 62, compare learning rates 0.0003 and 0.0001 against the four saved 0.001 reference fits. This adds eight fits, usually about 15–25 minutes at the previous GPU speed.
Raw input, original CNN, whole-participant validation, epoch limit, and macro-F1 checkpointing match the reference. This is a small diagnostic; it does not select a final configuration for all participants. Results go into a separate Drive folder.

## 1. Setup
Select a GPU runtime. The code is pinned to `v4-lr-short-screen-v1` for resume.

In [ ]:
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')
repo = '/content/teco-v4-short'
run_ref = 'v4-lr-short-screen-v1'
if not os.path.isdir(repo):
    subprocess.run(['git', 'clone', '-q', '--branch', run_ref, '--single-branch',
                    'https://github.com/parmisbathayan/reproduce-teco-results.git', repo], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert commit == subprocess.check_output(['git', 'rev-parse', run_ref + '^{commit}'], text=True).strip()
assert not subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip(), 'Use a clean checkout of the fixed revision.'
print('Ready. Git commit:', commit, flush=True)


## 2. Run or resume the eight new fits
This cell streams the training process's output into Colab. Each fit prints START, then SAVED when its files are written; a fit may be quiet for 1–3 minutes. Rerun after a disconnect to skip completed fits.

In [ ]:
base = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly'
out_dir = os.path.join(base, 'v4_lr_screen_JAY_ARM_45_62')
command = [sys.executable, '-u', '-B', 'src/v4_lr_short_screen.py',
           '--trt-dir', '/content/drive/MyDrive/Thesis/Data/TRT_Total',
           '--labels-csv', '/content/drive/MyDrive/Thesis/Data/teco_sentiment_labels_task1.csv',
           '--source-out-dir', os.path.join(base, 'v4_stability'),
           '--out-dir', out_dir, '--config', 'configs/v4_lr_short_screen.json']
print('Starting reference checks and eight-fit diagnostic. Live output follows:', flush=True)
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as process:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        print('Stopped. Completed fits remain saved.', flush=True)
        raise
if returncode:
    raise subprocess.CalledProcessError(returncode, command)


## 3. Display results
The comparison contains the four old reference rows plus the eight new rows. Keep both seeds visible: an overall average can hide another seed-45 collapse.

In [ ]:
import pandas as pd
from IPython.display import display
metrics = pd.read_csv(os.path.join(out_dir, 'fold_metrics.csv'))
print('New fits saved:', len(metrics), '/ 8')
display(pd.read_csv(os.path.join(out_dir, 'summary_by_lr_seed.csv')))
display(pd.read_csv(os.path.join(out_dir, 'comparison_with_reference.csv'))
        [['learning_rate', 'seed', 'participant', 'macro_f1', 'collapsed', 'stopping_epoch']])
